# Problem 2 · RAG retrieval and evaluation

Fill in the functions that still raise `NotImplementedError`. Leave the provided cells unchanged. Run the notebook from top to bottom.

This is a blank copy for a second attempt. After you finish, run the hidden-test cell. That cell carries the cases in packed form and writes them out before scoring, so a cloned copy still has them.

## Given

- `docs_df`: 36 documents with columns `doc_id`, `title`, `topic`, `text`. Encode `text`. `topic` is metadata.
- `queries_df`: 8 labeled queries with columns `query_id`, `query`, `relevant_doc_ids`. `relevant_doc_ids` is a list of strings. A query may have more than one relevant document.
- `embed_texts(texts) -> np.ndarray`, shape `[N, dim]`.
- `embed_query(q) -> np.ndarray`, shape `[dim]`.

Embeddings use `text-embedding-3-small`. Locally the key is read from `.env`. On Colab, add a secret named `OPENAI_API_KEY`. A read-only copy does not inherit that secret, so each copy needs its own secret or a pasted key. The key is not stored in this notebook. Vectors are cached at `.cache/rag_retrieval_eval_embeddings.npz`.

Scale: **N = 36, Q = 8, dim = 1536**.

## Task

Build a minimal retrieval pipeline and report Recall@10, Precision@10, Hit@10, and MRR@10. Each score is the unweighted mean over queries. Use only the first 10 retrieved ids. Count each document once. A relevant document at rank 11 or later does not count.

Cosine similarity:

\[
\mathrm{sim}(q, d) = \frac{q \cdot d}{\|q\| \, \|d\|}
\]

If `q` or a document row has L2 norm 0, that similarity is 0.

| Metric | Definition for one query |
| --- | --- |
| Recall@k | size of the intersection of the top k and the relevant set, divided by the number of relevant documents. Empty relevant set scores 0 |
| Precision@k | size of that intersection divided by **k**. The denominator is k even when fewer than k ids are returned |
| Hit@k | 1 if the top k contains any relevant document, otherwise 0 |
| MRR@k | 1/r when the first relevant document is at rank r (1-based) and r ≤ k, otherwise 0 |

Example: `retrieved = [d3, d1, d9]`, `relevant = {d1, d2}`, `k = 10`. `d1` is at rank 2 and `d2` is absent. Recall = 1/2, Precision = 1/10, Hit = 1, MRR = 1/2.

Keep the function names and signatures. Tests call them by name.

## Score (100)

| Area | Points | What is scored |
| --- | --- | --- |
| Logic | 70 | The retrieval and metrics below match the definitions |
| Performance | 15 | Work stays small for this N, Q, and dim, and the same approach still behaves well if N grows into the low thousands |
| Style | 15 | Names, control flow, and structure can be read aloud in an interview |

Logic is split as follows:

| Function | Points |
| --- | --- |
| `build_doc_embeddings` | 10 |
| `cosine_similarity` | 18 |
| `top_k_doc_ids` | 14 |
| `recall_at_k` | 5 |
| `precision_at_k` | 5 |
| `hit_at_k` | 4 |
| `mrr_at_k` | 6 |
| `evaluate` | 8 |


In [ ]:
import sys
import subprocess

if "google.colab" in sys.modules:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openai", "python-dotenv"])

import os
from getpass import getpass
from pathlib import Path
from typing import List

import dotenv
import numpy as np
import pandas as pd
import openai


def _resolve_openai_key() -> str:
    """Read the key without printing it.

    A Colab secret stays on the owner's notebook. Save a copy / playground
    starts without that secret and without a local .env file.
    """
    key = os.environ.get("OPENAI_API_KEY", "").strip()
    source = "environment"
    if not key and "google.colab" in sys.modules:
        from google.colab import userdata

        try:
            key = str(userdata.get("OPENAI_API_KEY")).strip()
            source = "Colab secret"
        except Exception:
            key = ""
    if not key:
        env_path = dotenv.find_dotenv(usecwd=True) or str(Path.cwd() / ".env")
        if Path(env_path).is_file():
            dotenv.load_dotenv(env_path)
            key = os.environ.get("OPENAI_API_KEY", "").strip()
            source = ".env"
    if not key and "google.colab" in sys.modules:
        key = getpass("OPENAI_API_KEY (not saved in the notebook): ").strip()
        source = "pasted key"
    if not key:
        raise RuntimeError(
            "OPENAI_API_KEY was not found. On Colab, add a secret named "
            "OPENAI_API_KEY, or paste the key when prompted."
        )
    os.environ["OPENAI_API_KEY"] = key
    print(f"OPENAI_API_KEY loaded from {source}.")
    return key


_resolve_openai_key()
client = openai.OpenAI()
EMBED_MODEL = "text-embedding-3-small"


In [ ]:
# Provided. Do not edit.
_EMB_CACHE: dict[str, np.ndarray] = {}
CACHE_PATH = Path(".cache") / "rag_retrieval_eval_embeddings.npz"


def _load_cache() -> None:
    if not CACHE_PATH.exists():
        return
    try:
        data = np.load(CACHE_PATH, allow_pickle=True)
        for key, vec in zip(data["keys"].tolist(), data["vecs"]):
            _EMB_CACHE[str(key)] = np.asarray(vec, dtype=np.float64)
    except Exception as exc:
        print(f"Ignoring a broken embedding cache: {exc}")


def _save_cache() -> None:
    CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    keys = np.array(list(_EMB_CACHE.keys()), dtype=object)
    vecs = np.stack(list(_EMB_CACHE.values())).astype(np.float64)
    np.savez(CACHE_PATH, keys=keys, vecs=vecs)


def embed_texts(texts: List[str]) -> np.ndarray:
    """Encode texts into a matrix of shape [N, dim]."""
    texts = [str(t) for t in texts]
    if len(texts) == 0:
        return np.zeros((0, 0), dtype=np.float64)

    missing = []
    seen = set()
    for text in texts:
        if text not in _EMB_CACHE and text not in seen:
            seen.add(text)
            missing.append(text)

    batch_size = 64
    for start in range(0, len(missing), batch_size):
        batch = missing[start : start + batch_size]
        response = client.embeddings.create(model=EMBED_MODEL, input=batch)
        ordered = sorted(response.data, key=lambda item: item.index)
        if len(ordered) != len(batch):
            raise RuntimeError("embedding response length does not match the input")
        for text, item in zip(batch, ordered):
            _EMB_CACHE[text] = np.asarray(item.embedding, dtype=np.float64)
        _save_cache()
        print(f"embedded {start + len(batch)}/{len(missing)}")

    return np.vstack([_EMB_CACHE[text] for text in texts])


def embed_query(q: str) -> np.ndarray:
    """Encode one query into a vector of shape [dim]."""
    return embed_texts([q])[0]


_load_cache()


In [ ]:
# Provided corpus and labeled queries. Do not edit.
DOCS = [
    ("d01", "Definition of cosine similarity", "retrieval", "After text is turned into vectors, cosine similarity is the dot product of the two vectors divided by the product of their L2 norms. It compares direction only. If either vector has norm 0, the similarity should be recorded as 0, so division by zero does not produce NaN."),
    ("d02", "Document matrix and the query vector", "retrieval", "Before retrieval, encode every document into a matrix D whose shape is the number of documents by the embedding dimension. Encode the user's question separately as a vector q. Compare q with each row of D."),
    ("d03", "Top-K retrieval", "retrieval", "After computing the similarity between the query and each document, take the Top-K doc_ids in descending score order. Those K documents are the context a RAG system passes to the generator."),
    ("d04", "Tomato and egg stir-fry", "cooking", "Tomato and scrambled egg: cut the tomatoes into chunks and beat the eggs with a pinch of salt. Scramble the eggs first and take them out as soon as they set. Then fry the tomatoes in oil until they release juice, return the eggs, add salt and a little sugar, and toss to finish."),
    ("d05", "Heat for tomato and egg", "cooking", "A home-style tomato and egg stir-fry should keep the eggs tender and the tomatoes saucy. Use little sugar and leave some juice. Scatter scallions before serving. Eggs left in the pan too long turn rubbery."),
    ("d06", "Kung pao chicken", "cooking", "Kung pao chicken uses chicken thigh, peanuts, dried chilies, and Sichuan peppercorns, seasoned with sugar, vinegar, and soy sauce. It is a Sichuan dish. The main ingredients are not tomato and egg."),
    ("d07", "Overview of Mars", "space", "Mars is the fourth planet from the Sun. Its surface looks red because of iron oxide. It is a terrestrial planet with a thin atmosphere, and there is water ice at the poles."),
    ("d08", "Mars exploration and water", "space", "The Martian atmosphere is mostly carbon dioxide. Landers such as Zhurong and Perseverance look for evidence of past water in ancient riverbeds and underground, because water makes a once-habitable environment more likely."),
    ("d09", "Jupiter's Great Red Spot", "space", "Jupiter is the most massive gas giant in the Solar System. Its famous feature is the Great Red Spot, a huge storm that has lasted for centuries. Jupiter is not Mars."),
    ("d10", "Traveling in basketball", "basketball", "Traveling is called when a player who is holding the ball lifts the pivot foot and puts that foot back down before the ball leaves the hand. The referee calls a traveling violation and the offense loses possession."),
    ("d11", "How the pivot foot is chosen", "basketball", "Under FIBA rules, a player may establish a pivot foot on the catch. If both feet are off the floor when the ball is caught, the first foot to land is the pivot foot. If both feet land at the same time, either foot may be the pivot foot."),
    ("d12", "Offer", "contract", "An offer is a statement of intent to enter a contract with another person. Its terms must be definite, and it must show that the offeror is bound by that statement once the offeree accepts."),
    ("d13", "Acceptance and formation", "contract", "Acceptance is the offeree's agreement to the offer. It takes effect when notice of acceptance reaches the offeror. When acceptance takes effect, the contract is formed, and both sides have the rights and duties they agreed to."),
    ("d14", "Withdrawal of an offer", "contract", "An offer may be withdrawn. Notice of withdrawal must reach the offeree before the offer does, or at the same time. After the offer has arrived it can no longer be withdrawn; it can only be revoked when the conditions for revocation are met."),
    ("d15", "Photosynthesis and the light reactions", "biology", "Photosynthesis takes place mainly in the chloroplast. The light reactions occur on the thylakoid membrane. They use light energy to split water, produce ATP and NADPH, and release oxygen."),
    ("d16", "The Calvin cycle", "biology", "The carbon reactions, also called the Calvin cycle, occur in the chloroplast stroma. They use the ATP and NADPH from the light reactions to fix carbon dioxide into sugar. Without that energy and reducing power, the carbon reactions cannot run."),
    ("d17", "Storming of the Bastille", "history", "On 14 July 1789, people in Paris stormed the Bastille. The event became the symbol that the French Revolution had begun, and France later made 14 July its national day."),
    ("d18", "Napoleon and the Civil Code", "history", "Napoleon later established an empire and oversaw the French Civil Code. That was a regime and a statute from after the French Revolution, not something that happened on the day the Bastille was stormed in 1789."),
    ("d19", "The Industrial Revolution", "history", "The Industrial Revolution began in the British textile industry in the eighteenth century, and machine production and the factory system gradually spread. It is a different stretch of history from the French Revolution."),
    ("d20", "Backpropagation", "dl", "When training a neural network, backpropagation uses the chain rule to send the partial derivative of the loss with respect to the output back through each layer, producing a gradient for every weight."),
    ("d21", "Updating weights by gradient descent", "dl", "Once the gradients are known, stochastic gradient descent sets a weight to w minus the learning rate times the gradient. Too large a learning rate makes the loss oscillate or diverge. Too small a learning rate converges slowly."),
    ("d22", "Loss and gradients", "dl", "Classification often uses cross-entropy, and regression often uses mean squared error. Backpropagation computes the gradient of these losses with respect to the parameters, and the optimizer then updates the network from that gradient."),
    ("d23", "The Great Wall", "misc", "The Great Wall is an ancient Chinese military fortification. The Ming wall people know today runs from around Shanhai Pass in the east to Jiayu Pass in the west."),
    ("d24", "Espresso", "misc", "Espresso forces water at about 90 degrees Celsius through finely ground coffee at high pressure for a short time, producing a very small, very strong shot."),
    ("d25", "The piano", "misc", "A modern piano usually has 88 keys. Pressing a key makes a hammer strike a string, and the soundboard amplifies the vibration."),
    ("d26", "Pythons", "misc", "A python is a large nonvenomous snake that coils around its prey until the prey cannot breathe. The name is easy to associate with the Python programming language, but the animal is a different thing."),
    ("d27", "Fuxing high-speed trains", "misc", "Fuxing is a Chinese high-speed electric trainset. It runs on dedicated track and can cruise at 350 kilometers per hour."),
    ("d28", "Mount Everest", "misc", "Mount Everest stands in the Himalayas and is the highest mountain above sea level. The south side is in Nepal and the north side is in Tibet, China."),
    ("d29", "Honey", "misc", "Honey is what bees make after collecting nectar and processing it repeatedly. Its main sugars are fructose and glucose, and the low water content lets it keep for a long time."),
    ("d30", "Hamlet", "misc", "Shakespeare's tragedy Hamlet follows a prince of Denmark who, after learning that his uncle killed the king, cannot bring himself to take revenge."),
    ("d31", "The Mariana Trench", "misc", "The Pacific is the largest ocean on Earth. The Mariana Trench lies in the western Pacific and is the deepest known ocean trench."),
    ("d32", "Pencils", "misc", "Pencil lead is mostly graphite and clay. Graphite particles stay on the paper while writing, so the mark looks gray-black."),
    ("d33", "Typhoons", "misc", "A typhoon is a strong tropical cyclone in the northwest Pacific. It often reaches the coast of East Asia in summer and autumn, bringing high wind and heavy rain."),
    ("d34", "Checkmate", "misc", "In chess, if the king is under attack and no legal move can remove that attack, the position is checkmate and the game is over."),
    ("d35", "Yogurt", "misc", "Yogurt is milk fermented by lactic acid bacteria. Some of the lactose is turned into lactic acid, so the taste becomes sour and the texture thickens."),
    ("d36", "Rainbows", "misc", "A rainbow is a colored arc formed when sunlight enters water droplets, is refracted and reflected, and splits into separate colors."),
]

QUERIES = [
    {
        "query_id": "q01",
        "query": "After computing cosine similarity between a query vector and each document embedding, how do you select the closest Top-K documents?",
        "relevant_doc_ids": ["d01", "d02", "d03"],
    },
    {
        "query_id": "q02",
        "query": "How do you cook a home-style tomato and egg stir-fry, and what order and heat should you use for the eggs and tomatoes?",
        "relevant_doc_ids": ["d04", "d05"],
    },
    {
        "query_id": "q03",
        "query": "Which planet from the Sun is Mars, and why do exploration missions look for water?",
        "relevant_doc_ids": ["d07", "d08"],
    },
    {
        "query_id": "q04",
        "query": "How is traveling called in basketball, and how is the pivot foot chosen?",
        "relevant_doc_ids": ["d10", "d11"],
    },
    {
        "query_id": "q05",
        "query": "How is a contract formed? What do offer, acceptance, and withdrawal of an offer mean?",
        "relevant_doc_ids": ["d12", "d13", "d14"],
    },
    {
        "query_id": "q06",
        "query": "Where in the chloroplast does photosynthesis happen, and what do the light reactions and the carbon reactions each do?",
        "relevant_doc_ids": ["d15", "d16"],
    },
    {
        "query_id": "q07",
        "query": "On what date was the Bastille stormed during the French Revolution, and what did that event symbolize?",
        "relevant_doc_ids": ["d17"],
    },
    {
        "query_id": "q08",
        "query": "When training a neural network, how are the weights updated after backpropagation has produced the gradients?",
        "relevant_doc_ids": ["d20", "d21", "d22"],
    },
]

docs_df = pd.DataFrame(DOCS, columns=["doc_id", "title", "topic", "text"])
queries_df = pd.DataFrame(QUERIES)


In [ ]:
docs_df.head(3)


In [ ]:
queries_df[["query_id", "query", "relevant_doc_ids"]]


## 1. `build_doc_embeddings`


In [ ]:
def build_doc_embeddings(docs_df: pd.DataFrame):
    """Return document embeddings aligned with doc ids."""
    raise NotImplementedError("build_doc_embeddings")


# Provided check. Run this after you implement the function.
D, doc_ids = build_doc_embeddings(docs_df)
D = np.asarray(D, dtype=np.float64)
doc_ids = np.asarray(doc_ids).astype(str)
assert D.ndim == 2, "D must be a 2-D matrix"
assert D.shape[0] == len(docs_df) == len(doc_ids), "D, doc_ids, and docs_df must have the same length"
assert list(doc_ids) == docs_df["doc_id"].astype(str).tolist(), "doc_ids must follow the row order of docs_df"
print("D.shape =", D.shape)


## 2. `cosine_similarity`


In [ ]:
def cosine_similarity(q: np.ndarray, D: np.ndarray) -> np.ndarray:
    """Cosine similarity between q and each row of D."""
    raise NotImplementedError("cosine_similarity")


## 3. `top_k_doc_ids`


In [ ]:
def top_k_doc_ids(q: np.ndarray, D: np.ndarray, doc_ids, k: int = 10):
    """Top-k document ids for one query vector."""
    raise NotImplementedError("top_k_doc_ids")


## 4. Metrics and `evaluate`


In [ ]:
def recall_at_k(retrieved, relevant, k: int = 10) -> float:
    """Recall@k for one query."""
    raise NotImplementedError("recall_at_k")


def precision_at_k(retrieved, relevant, k: int = 10) -> float:
    """Precision@k for one query."""
    raise NotImplementedError("precision_at_k")


def hit_at_k(retrieved, relevant, k: int = 10) -> float:
    """Hit@k for one query."""
    raise NotImplementedError("hit_at_k")


def mrr_at_k(retrieved, relevant, k: int = 10) -> float:
    """MRR@k for one query."""
    raise NotImplementedError("mrr_at_k")


def evaluate(queries_df: pd.DataFrame, D: np.ndarray, doc_ids, k: int = 10) -> dict:
    """Macro-averaged Recall@k, Precision@k, Hit@k, and MRR@k."""
    raise NotImplementedError("evaluate")


## Hidden tests

Run the next cell. It prints the objective logic score out of 70. Performance and style are graded separately.


In [ ]:
# Provided. Do not edit.
import base64
import importlib
import sys
import zlib
from pathlib import Path

_HIDDEN_TESTS_B64 = """
eNrFGmtv4zbye4D8B0LFYaVWUe3so1vjfCjQ9K5Fe22x12+CoFUsyuFFlhRJ3iYN/N9vZkhKpEx5c83iblvENufJ4XBmOORn
7OLzC7apc1FtV2zfFxdvceT8zPO870We84ptbvjmtmNF3bI226Yt71vBP2RlSn+aNtv0YsMj0TxU19H52fnZbzec6WFW1T2/
rutbVlflA9tkZdmxdl/5QcR+5Lxh/Y0A3qLkbFPWHc/ZvupFCcP8/Oy6zCqQnLUcMUqeR6TX+VnR1juWpsW+37c8TZnYNXXb
s6wCaVkv6qpDPdRotd81DyzrWNUMY01W5TAC/zc5op6f5bxg6abuRMX9u5BdBavzMwb/7tgaCKOsy9o2e0BQ3j80fA1jRVln
/ZtXQdTy7iZruH+xDCTRlU105SCSiGBfmAkhl6LKym1U1e0OCbJ70a2XAft8CrtTpJ3YdZLyD97WnX8VkRLxIpmVBgO5+CBy
7l+xbxhOBcWHrN73a2QXst9veMvXUqu/sYWigyXftxVJNIyVdWnXt2kput4HT9jzTttM4ccAlpCAvIe+MlExkySRHD9jv1z/
m4PLfADfqUXVdxH7lbdAtssqcCN/+TqA9c1Z1z+U+mdL7gWz2MCccoa6g4ekP/3yjx++TX/95Yeff/sXWOhRKuVd70WZp3m9
Sfnumufo8Z23YstFqBDk6qcwTVFmregfEPpWQ/u6SW+JXORE90pDWo5unWZ9egvjr/VwA+OiA2c8gtyIXo8NTHZtq8fe6DHc
YPus5zCGahyGuX37/Xff/viEucUSbKD0/L7vUlGlbf17Wrc5b73wCElOchZMfuYYB5Zdusv6zY0Uo1GSUwY2VdTwI/4KIF3G
BbmTRF2abVvupK2bBr70bljb39TbGvaYC4rbC+c2C7vb8/bhaLJTfzEnCrDbVHS0d9K6wH1kcSf4sflp+Dbd9ml1PA4W3XKb
KHF7qKmJgrQQaJfLlN9vyj1sJYu7QuG7pn+A6F+CV1a9CyPfN6XYgL8eYSWze8LUZQRSAIKY19ctmunWkmagGQIpKY2aJ469
FtOP6VQTxw6M6UfHN3WVp02ZbcBtFELR8xZwEsceNeeih1PKeCnPYFNYfmJj3fKHzg15R7b95tIN/VXbYg7he9HPgf757t0I
SmR8wf82ZdZ1LH23ryrernSegoAPYQPsl/odL4uQVUO4p2wEY1FFCambjBaZKCFNIyxOJrAGREHkXrOFE5BW2Y4IO977ASqn
laGiRGsCSCHDtcKM1oO4teeZyomCoMbIVIEv1mw5C5VaRFme+/gtsBFlshvHALXLthwtAcgoW+Yo1IvxsoNaxntE0GHFHuXw
wZsxWZQ1Da9yX/G0LFBxnhsGMOdbVGQyWpFoC6abqK10QsfKrkvuF1Xgss1Ui1HvYl9tsMxiULvJ2RUQDXPPaRr2c13xEaAG
i8ooJXoOcVAVXy35nVZIDsJ05HBEs3YkEiWZVpooQDOUu5oKVhVKtZW1E0SDvN5FoEW2LyE4VFt/aVZxMEBlV1b6nfiDr/2v
QvY6UAh3mBZcSEsLqSizHrAI+7hW7NsHQ8lt3SumRv2oy1KEYGk6W+BpDkqiiwWCPsqD3zdQi9HGTI8oJRrET9707Dv6QFeA
WhrGTHvLFTv2IrV60vSwCVAPH2iDKKWtlqa4NWDgYDqUtXyStQwCk7JBm1CWw2y9ZrBmMN3CkwPgGI+wJ7IeilOFGrIXBHsR
ks8Eg1ynHF2FkHnLko4tIydtupC1fV2ul/ziDRTz6uvb4BTjaRHjFqDXd16AFKELHuUH5AVxfHEZLUK2iPCUEC/w+0vjO44P
xwdyC6eTjkePiXuNggYxYHmtybzPPd+ZOrGtuk/nS2O1SJM1fOllKM8hMAfRyaVBDDp2XSyjRUDL1tZ70E6eqV6fXHSz9jxi
ukzIiE/l+bTalYQUmMullACdzHfP6lIrYJHbmijY4Hi2r8gKec5b5NH18n/kJGO1/nxPecpZYJz/nAOpZZBIk4UYtr6Ezi+E
httLoVIrlrtQwsAZhNoaIYNvOrnSYUHHh20HruhfWD2QkN2KKl97Ow5nC4R7dkcgBmaxSOh0L/BkTxyTaWbH88kkr+PQNKvb
J6YxoxPu6XwO7aI+64ZGStdAye5DrHsdssvoMmRf242Zrs9gCWP4BrP1JbHcZDB3/TsZmjB2J0gGuOUQ4JSeORW4hZc/ioM3
WgTsv+V+yQe2QTKT94GcDDUsE1h//QmcfzxM/imvN2sBpzPFq1eJK8A5D7mwjrA8PfZ0MKWF1AuSLo/OT+rBjtgFuOBAIQ0J
A2hLIIBVIhz8GsxKVadgu0eFJLgDxwxdeGj2xyOsw5jG0TbyGxlormQD3e3lI15oHVzEy8UnWkV59n/WMipV3Ut5Os7NtB/s
FUX+5qpOjYtwaxmIwDi7uNeEyJzrgpCDN01CJwt8HQ0qWBzAWr5dwF6+fK3OoLBhSzp0Ylm/rwT2IP0LYNUIChH4QWV+pYOK
7L24QovkNYYW9XsaWuAYBidDK74UXvMoVotX+XEwqYJk9uygNRlcUY6ESgR64+0nckaj4fQ8j9Q6T33SVj2CEhedIYCAc6sC
zg3PMC69eYLfOtpj+O/IzwjPdlCphumihWg7cFIUf2Buf5V84hXiJG7HJRSN4TlyNyXPHba2Nt0kf8qW2zSDmn0+ZfWhWzbF
nbTiFDp0yKaIQwdNoUAbbIoytM7GpI1tAYzosdQpHBUJUUiIbJJg7nhOjiDz6b0rny4vdRrVrUZE9u6XC2zU3b/0nEnWzhRz
rU+zCJY4vtQnHISFcGsQYPof6vBj9q5eo8UbEAbGqPoSmoqa72Ker6MzaTAF6Jy2S/Yle3War+5uzrKUJn6Snqe6uSb7Ac+P
aeEcVl7OS5lpTjvWELiT5vGTtD/R0nbyztDv6M81CdHfaVRLXD7RXq629pzJskFELD9e0dQuB898RphXoeej0d0OWLrBPIlY
engaO4YG+hg7BtTTRb+8TIZLMK+C7vvioA81YovXpYYc2QT16GpMndCsVqoB8PHmygxLJCQGCcmkSayOQO4TgZWQKqA3ZSeg
mvHbvjJWfZrltE2zlN8vlif7NOrwRMEwp1iYL+nvpY6IKFJQedPk0VXWZ39vYT2NdPlo93A90hGY0rXIHbnbHfn13Wa4BbGR
CbPCxgaidXC7TF+ua/g8otD7yrwpI93Jo1Hv0JxLYjI4DCnzONYr30VTa39UU6dqV0mj6vz5G0WL+FN1UIHeWj7Y9xWDF6sO
+2QbfMS7NLV2ROMMZyyvN9wtreBCPVqwL9Cx4C/GKUgW0Bk0L47MuyYggOT3EQJ59/Qk3vIu6mOoB9eB78RNm7l5oaB7iSc+
w1Tsd1hhRMGqTKK+qF4kB9YLuHCZ1DLGKUe5FjiR2FixYsY/7Gu+lY4c6jmEXN+MmM25yE50IB5PNDGwoGIIP6EcGlYW1MQh
UrXS3p+cNJi8dCQSJQANpGU9qi+HiSHUsHndJNUJxwceWqsIT+6dH7g9V+oz7p70EbgcJsWHnAhOO1ECguHgbgIdiYheJUyy
EI1NU5Dz9cQ4a0lzOhd1nFeUiuTLBwhiyRPzkSRw5CMCUD6yblppAD0BX9dQtqIFwG9oegJb16y8ipVWSQQf6JGSqTETeLmB
LGPKJbJbhZzR1DTS7Xc+nJ38zY2UtrnRsgL2FyhuFosgOamGkS1VExhlOnu9M7lTTWHInfR7yHr/RUKToZ+SFKQXTCtDcpkm
p170Jd3sez1h9oTZuzDrRmwI84+hEtw50EBpwtpmu50s3crmRtaK0B305rObmiN+RKKsN3EMbU2sCxLnUWdMcUAkNwKSPj/V
ufbK/zftjY4xTXtPfoaHB3ZpKuscrywYTPoVEhPNGWtnSkAImQBog6FTMaHTO1dSkiucorOj9vzzLXuLW50LGpsgnOBuv/sa
rbK25j4ATnDSd7FX1sUHw4seffd6hbWhQqNb1zHgHlHJl3+np7k6vr5Or+xo4dsUx0ex007lPrfNvX2zrnCuQkOpwMxU9MKx
T+WbR/VaUvd48HaHijZ5z0NA2PGjobJr6AsprAuFpX4HAfsrgxvhr48SFrbVfZDmT/Dtqx147KEgq2hZ4AuVQeWmBeq01u81
025Tt/zofqenlAeVnO5DoUhvfOVJVMwv663YBDpIYAqRz3mkQTCVWO85HRWFep+8ZtbjyBjZJNMLFsxjS5XBgArZK3JRjENq
gc3XP4avSMXXKjNKPQOkJxFrhtlTMg3kex81lc8J4UsTbmR1shecMIn7OC7NVniQxvSboYnLEEFw+PJR/j6MjT9J+RMa+JiM
BALZVwubwHvqq1tRyaZWZBZe+LKb7IUXcHqR4Mk2PCbDOzro/JlPyrNthkX18Jyo0w+JcBHevx8YvX8f0bPvcfPhassHaoY4
ieB6TGRCzMtIc3zSZzVB046GCbOKTNv3LTeC8wwu/CSvjlt5Algdu4B6jo9S4dm6ZPpoyQAnINlWzkVn12/SRt92yDFlXWBx
TTQDL3TlT6PV6QBiRSF0tInG52f/AVOKlGk=
"""


def _materialize_hidden_tests() -> None:
    """Write the packed checks next to the notebook.

    Locally, an existing rag_retrieval_eval_hidden_tests.py is left in place.
    On Colab the runtime disk is empty after a clone, so this cell writes the
    file and marks it read-only. The plaintext cases are not in this cell.
    """
    path = Path("rag_retrieval_eval_hidden_tests.py")
    in_colab = "google.colab" in sys.modules
    if path.is_file() and not in_colab:
        return
    if path.exists():
        try:
            path.chmod(0o644)
        except OSError:
            pass
    path.write_bytes(zlib.decompress(base64.b64decode(_HIDDEN_TESTS_B64)))
    try:
        path.chmod(0o444)
    except OSError:
        pass


_materialize_hidden_tests()
if "rag_retrieval_eval_hidden_tests" in sys.modules:
    importlib.reload(sys.modules["rag_retrieval_eval_hidden_tests"])
else:
    import rag_retrieval_eval_hidden_tests

rag_retrieval_eval_hidden_tests.run(globals())


## Metrics

Run the next cell after the functions are implemented.


In [ ]:
metrics = evaluate(queries_df, D, doc_ids, k=10)
print(f"Recall@10 = {metrics['Recall@10']:.4f}")
print(f"Hit@10 = {metrics['Hit@10']:.4f}")
print(f"MRR@10 = {metrics['MRR@10']:.4f}")
print(f"Precision@10 = {metrics['Precision@10']:.4f}")


## Afterward

When the hidden tests pass, come back and ask for a score. The score covers logic, performance, and style.
